# Semantic Search

**Multilingual Sentence Transformer-Based Semantic Retrieval**

This notebook implements semantic similarity retrieval for graduation projects using a pretrained multilingual Sentence Transformer and Cosine Similarity.

The system supports Arabic and English project ideas and retrieves the most semantically related historical projects.

## 1. Setup and Dependencies

Install and import the required libraries for semantic retrieval.

In [1]:
%pip install -q sentence-transformers

In [2]:
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Libraries imported successfully!
Device: cuda


## 2. Load and Validate Dataset

Load the cleaned graduation projects dataset from GitHub and verify its structure.

In [3]:
# Load dataset
url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(url, encoding="utf-8-sig")

# Standardize columns
df.columns = df.columns.str.strip().str.lower()
df["department"] = df["department"].str.strip()

# Validate dataset
assert df.shape == (103, 7)
assert df["project_id"].is_unique
assert df[["title", "abstract"]].notna().all().all()
assert df[["title", "abstract"]].apply(
    lambda col: col.astype(str).str.strip().ne("").all()
).all()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Departments:", df["department"].nunique())

display(df.head(3))

Dataset loaded successfully!
Shape: (103, 7)
Departments: 6


,project_id,title,abstract,department,year,keywords,source_page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7


## 3. Text Preparation

Combine project titles and abstracts into a single text representation while preserving semantic information.

In [4]:
# Prepare project texts
semantic_df = df.copy()

semantic_df["text"] = (
    semantic_df["title"].str.strip()
    + ". "
    + semantic_df["abstract"].str.strip()
)

semantic_df["text"] = (
    semantic_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

project_texts = semantic_df["text"].tolist()

print("Total project texts:", len(project_texts))
print("\nSample project text:")
print(project_texts[0][:500])

Total project texts: 103

Sample project text:
Automatic Plant Watering and Tracking System. Many people struggle with watering their plants regularly, which usually leads to plants becoming unhealthy or dying. To address this common issue, it is important to monitor basic factors such as soil moisture and temperature in order to determine the proper time and amount of water needed. Since manual watering is not always consistent or reliable, this automatic plant watering system is designed to assist gardeners by providing an accurate irrigat


## 4. Load Multilingual Sentence Transformer

Load a pretrained multilingual Sentence Transformer to generate semantic text embeddings for Arabic and English project descriptions.

In [5]:
# Load pretrained model
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

print("Model loaded successfully!")
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda
Embedding dimension: 768


/tmp/ipykernel_424/400151206.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 5. Generate Project Embeddings

Encode the 103 graduation project descriptions into normalized dense vectors for semantic similarity retrieval.

In [6]:
# Generate embeddings
project_embeddings = model.encode(
    project_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Validate embeddings
assert project_embeddings.shape == (
    len(project_texts),
    model.get_sentence_embedding_dimension()
)

assert np.isfinite(project_embeddings).all()

print("Embeddings generated successfully!")
print("Embeddings shape:", project_embeddings.shape)
print("Data type:", project_embeddings.dtype)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings generated successfully!
Embeddings shape: (103, 768)
Data type: float32


/tmp/ipykernel_424/265478390.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 5.1. Token Length Diagnostic

Analyze project token lengths to identify descriptions that exceed the Sentence Transformer's maximum sequence length.

In [14]:
# Check model token limit
max_tokens = model.max_seq_length

# Count tokens without truncation
token_counts = [
    len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )
    for text in project_texts
]

token_diagnostics = pd.DataFrame({
    "project_id": semantic_df["project_id"],
    "token_count": token_counts
})

token_diagnostics["is_truncated"] = (
    token_diagnostics["token_count"] > max_tokens
)

token_diagnostics["tokens_exceeding_limit"] = (
    token_diagnostics["token_count"] - max_tokens
).clip(lower=0)

print("SEMANTIC MODEL TOKEN DIAGNOSTIC")
print("=" * 55)

print("Model maximum tokens:", max_tokens)
print("Total projects:", len(token_diagnostics))
print("Projects exceeding limit:", token_diagnostics["is_truncated"].sum())
print(
    "Percentage exceeding limit:",
    round(token_diagnostics["is_truncated"].mean() * 100, 2),
    "%"
)
print("Average tokens:", round(token_diagnostics["token_count"].mean(), 2))
print("Maximum tokens:", token_diagnostics["token_count"].max())

print("\nTOP 10 LONGEST PROJECTS")
display(
    token_diagnostics.sort_values(
        "token_count", ascending=False
    ).head(10)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (210 > 128). Running this sequence through the model will result in indexing errors


SEMANTIC MODEL TOKEN DIAGNOSTIC
Model maximum tokens: 128
Total projects: 103
Projects exceeding limit: 98
Percentage exceeding limit: 95.15 %
Average tokens: 270.09
Maximum tokens: 431

TOP 10 LONGEST PROJECTS


,project_id,token_count,is_truncated,tokens_exceeding_limit
95,P096,431,True,303
74,P075,430,True,302
51,P052,424,True,296
81,P082,422,True,294
5,P006,408,True,280
73,P074,395,True,267
54,P055,391,True,263
94,P095,383,True,255
31,P032,382,True,254
4,P005,372,True,244


## 5.2. Token-Based Text Chunking

Split long project descriptions into overlapping token-based chunks that fit within the model's maximum sequence length.

This approach preserves information from the full project descriptions while retaining contextual overlap between adjacent chunks.

In [16]:
# Configure chunking
max_tokens = model.max_seq_length
special_tokens = model.tokenizer.num_special_tokens_to_add(pair=False)

chunk_size = max_tokens - special_tokens
chunk_overlap = 20
chunk_step = chunk_size - chunk_overlap

assert chunk_step > 0

# Split text into token chunks
def chunk_text(text):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    chunks = []

    for start in range(0, len(token_ids), chunk_step):
        chunk_ids = token_ids[start:start + chunk_size]
        chunks.append(chunk_ids)

        if start + chunk_size >= len(token_ids):
            break

    return chunks


# Generate project chunks
project_chunks = [
    chunk_text(text) for text in project_texts
]

chunk_counts = [len(chunks) for chunks in project_chunks]

# Validate token lengths
assert all(
    len(chunk) + special_tokens <= max_tokens
    for chunks in project_chunks
    for chunk in chunks
)

print("Token-based chunking completed successfully!")
print("Total projects:", len(project_chunks))
print("Total chunks:", sum(chunk_counts))
print("Average chunks per project:", round(np.mean(chunk_counts), 2))
print("Maximum chunks per project:", max(chunk_counts))
print("Chunk size:", chunk_size)
print("Overlap:", chunk_overlap)

Token-based chunking completed successfully!
Total projects: 103
Total chunks: 298
Average chunks per project: 2.89
Maximum chunks per project: 4
Chunk size: 126
Overlap: 20


## 5.3. Generate Chunk Embeddings

Generate normalized embeddings for all token-based chunks while preserving the model's sequence length limit.

In [20]:
# Create safe text chunks
def safe_chunk_text(text, overlap=20):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    chunks = []
    start = 0

    while start < len(token_ids):
        end = min(
            start + model.max_seq_length - 2,
            len(token_ids)
        )

        # Ensure re-tokenized text fits the model
        while end > start:
            chunk = model.tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True
            )

            length = len(model.tokenizer.encode(
                chunk,
                add_special_tokens=True,
                truncation=False
            ))

            if length <= model.max_seq_length:
                break

            end -= 1

        if end == start:
            raise ValueError("Unable to create a valid chunk.")

        chunks.append(chunk)

        if end == len(token_ids):
            break

        start = max(start + 1, end - overlap)

    return chunks


# Generate safe chunks
project_chunk_texts = [
    safe_chunk_text(text)
    for text in project_texts
]

all_chunk_texts = [
    chunk
    for chunks in project_chunk_texts
    for chunk in chunks
]

# Validate chunk lengths
assert all(
    len(model.tokenizer.encode(
        chunk, add_special_tokens=True
    )) <= model.max_seq_length
    for chunk in all_chunk_texts
)

# Generate embeddings
chunk_embeddings = model.encode(
    all_chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

assert chunk_embeddings.shape == (
    len(all_chunk_texts),
    model.get_sentence_embedding_dimension()
)

assert np.isfinite(chunk_embeddings).all()

print("Chunk embeddings generated successfully!")
print("Total projects:", len(project_chunk_texts))
print("Total chunks:", len(all_chunk_texts))
print("Embeddings shape:", chunk_embeddings.shape)
print("Data type:", chunk_embeddings.dtype)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Chunk embeddings generated successfully!
Total projects: 103
Total chunks: 298
Embeddings shape: (298, 768)
Data type: float32


/tmp/ipykernel_424/1584731939.py:80: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 5.4. Aggregate Chunk Embeddings

Aggregate chunk embeddings using mean pooling to generate one normalized embedding per graduation project.

In [21]:
# Aggregate chunk embeddings per project
chunked_project_embeddings = []

start = 0

for chunks in project_chunk_texts:
    count = len(chunks)

    project_vectors = chunk_embeddings[start:start + count]

    mean_embedding = project_vectors.mean(axis=0)

    norm = np.linalg.norm(mean_embedding)

    if norm == 0:
        raise ValueError("Zero-norm project embedding.")

    chunked_project_embeddings.append(mean_embedding / norm)

    start += count

chunked_project_embeddings = np.vstack(
    chunked_project_embeddings
)

# Validate aggregated embeddings
assert start == len(chunk_embeddings)
assert chunked_project_embeddings.shape == (103, 768)
assert np.isfinite(chunked_project_embeddings).all()

assert np.allclose(
    np.linalg.norm(chunked_project_embeddings, axis=1),
    1.0,
    atol=1e-5
)

print("Project embeddings aggregated successfully!")
print("Shape:", chunked_project_embeddings.shape)
print("Data type:", chunked_project_embeddings.dtype)

Project embeddings aggregated successfully!
Shape: (103, 768)
Data type: float32


## 5.5. Original vs. Chunked Semantic Retrieval

Compare retrieval rankings before and after chunking using the same project idea and cosine similarity.

In [22]:
# Test query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

query_text = f"{test_title}. {test_description}"

# Encode query
query_embedding = model.encode(
    [query_text],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Compare similarity scores
original_scores = cosine_similarity(
    query_embedding, project_embeddings
)[0]

chunked_scores = cosine_similarity(
    query_embedding, chunked_project_embeddings
)[0]

def display_top5(scores, label):
    top_indices = np.argsort(-scores)[:5]

    results = semantic_df.iloc[top_indices][
        ["project_id", "title"]
    ].copy()

    results.insert(0, "rank", range(1, 6))
    results["similarity_score"] = scores[top_indices]

    print(label)
    display(results.reset_index(drop=True).round(4))

display_top5(original_scores, "Original Semantic Search")
display_top5(chunked_scores, "Chunked Semantic Search")

Original Semantic Search


,rank,project_id,title,similarity_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,0.8417
1,2,P080,A Smart Mobile Application For Skin Cancer Det...,0.5138
2,3,P089,AI-Powered Platform for Plant Identification a...,0.4956
3,4,P005,Thabat – ثبات,0.4779
4,5,P082,Predicting Diabetes Patients Based On Machine ...,0.4676


Chunked Semantic Search


,rank,project_id,title,similarity_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,0.8100
1,2,P080,A Smart Mobile Application For Skin Cancer Det...,0.5332
2,3,P005,Thabat – ثبات,0.5147
3,4,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,0.5130
4,5,P082,Predicting Diabetes Patients Based On Machine ...,0.4944


## 5.6. Load Development Evaluation Queries

Load the development evaluation queries to compare original and chunked semantic retrieval under consistent conditions.

In [23]:
# Load evaluation dataset
eval_url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(eval_url, encoding="utf-8-sig")

# Select development queries
dev_queries = (
    eval_df[eval_df["split"] == "dev"]
    .copy()
    .reset_index(drop=True)
)

assert len(dev_queries) == 40
assert dev_queries["query_id"].is_unique

print("Total development queries:", len(dev_queries))
print("\nLanguage distribution:")
print(dev_queries["language"].value_counts())

display(dev_queries.head(3))

Total development queries: 40

Language distribution:
language
en    20
ar    20
Name: count, dtype: int64


,query_id,query_title,query_description,language,generation_intent,reference_project_ids,split
0,Q003,Blood Stock Planning Desk,Hospital blood-bank staff need earlier notice ...,en,near_overlap,P059,dev
1,Q004,Everyday Sound Notifier,Deaf residents may miss a doorbell or househol...,en,near_overlap,P074,dev
2,Q006,مؤشرات تحقق المخرجات الدراسية,نقترح منصة لأعضاء هيئة التدريس تساعدهم على متا...,ar,near_overlap,P002,dev


## 5.7. Batch Retrieval Comparison

Compare original and chunked semantic retrieval across all 40 development queries using identical inputs and Top-5 rankings.

In [24]:
# Generate embeddings for development queries
dev_queries["query_text"] = (
    dev_queries["query_title"].str.strip()
    + ". "
    + dev_queries["query_description"].str.strip()
)

query_embeddings = model.encode(
    dev_queries["query_text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Calculate similarity matrices
original_similarities = cosine_similarity(
    query_embeddings, project_embeddings
)

chunked_similarities = cosine_similarity(
    query_embeddings, chunked_project_embeddings
)

# Retrieve Top-5 for each query
def batch_retrieve(similarity_matrix, method):
    results = []

    for i, query in dev_queries.iterrows():
        top_indices = np.argsort(-similarity_matrix[i])[:5]

        for rank, idx in enumerate(top_indices, start=1):
            results.append({
                "query_id": query["query_id"],
                "language": query["language"],
                "method": method,
                "rank": rank,
                "project_id": semantic_df.iloc[idx]["project_id"],
                "similarity_score": similarity_matrix[i, idx]
            })

    return pd.DataFrame(results)


original_batch_results = batch_retrieve(
    original_similarities, "original"
)

chunked_batch_results = batch_retrieve(
    chunked_similarities, "chunked"
)

# Validate results
assert original_batch_results.shape[0] == 200
assert chunked_batch_results.shape[0] == 200

print("Batch retrieval completed successfully!")
print("Original results:", len(original_batch_results))
print("Chunked results:", len(chunked_batch_results))

display(chunked_batch_results.head(10).round(4))

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batch retrieval completed successfully!
Original results: 200
Chunked results: 200


,query_id,language,method,rank,project_id,similarity_score
0,Q003,en,chunked,1,P059,0.7220
1,Q003,en,chunked,2,P083,0.5121
2,Q003,en,chunked,3,P017,0.4070
3,Q003,en,chunked,4,P070,0.4018
4,Q003,en,chunked,5,P082,0.3907
5,Q004,en,chunked,1,P074,0.6598
6,Q004,en,chunked,2,P003,0.5552
7,Q004,en,chunked,3,P036,0.5506
8,Q004,en,chunked,4,P069,0.4527
9,Q004,en,chunked,5,P085,0.3705


## 5.8. Reference Retrieval Comparison

Compare original and chunked semantic retrieval using reference project IDs from the development queries.

Reference Hit@5 is used as a preliminary diagnostic, not as a substitute for human relevance evaluation.

In [25]:
# Evaluate reference retrieval
def reference_hit_analysis(results, queries):
    records = []

    for _, query in queries.iterrows():
        refs = query["reference_project_ids"]

        if pd.isna(refs) or not str(refs).strip():
            continue

        reference_ids = {
            ref.strip()
            for ref in str(refs).split(";")
            if ref.strip()
        }

        retrieved_ids = set(
            results.loc[
                results["query_id"] == query["query_id"],
                "project_id"
            ]
        )

        records.append({
            "query_id": query["query_id"],
            "language": query["language"],
            "reference_hit_at_5": int(
                bool(reference_ids & retrieved_ids)
            )
        })

    return pd.DataFrame(records)


original_hits = reference_hit_analysis(
    original_batch_results, dev_queries
)

chunked_hits = reference_hit_analysis(
    chunked_batch_results, dev_queries
)

# Compare by language
comparison = pd.concat([
    original_hits.assign(method="Original"),
    chunked_hits.assign(method="Chunked")
])

summary = (
    comparison
    .groupby(["language", "method"])
    .agg(
        queries=("query_id", "count"),
        hits=("reference_hit_at_5", "sum"),
        hit_rate=("reference_hit_at_5", "mean")
    )
    .reset_index()
)

summary["hit_rate"] = (
    summary["hit_rate"] * 100
).round(2)

display(summary)

print("Reference retrieval comparison completed!")

,language,method,queries,hits,hit_rate
0,ar,Chunked,13,13,100.00
1,ar,Original,13,13,100.00
2,en,Chunked,13,11,84.62
3,en,Original,13,11,84.62


Reference retrieval comparison completed!


## 5.9. Reference Ranking Analysis

Compare the ranking positions of reference projects using Mean Reciprocal Rank at 5 (MRR@5).

Reference-based MRR@5 is a preliminary diagnostic and does not replace human relevance evaluation.

In [26]:
# Calculate reference-based reciprocal ranks
def reference_rank_analysis(results, queries):
    records = []

    for _, query in queries.iterrows():
        refs = query["reference_project_ids"]

        if pd.isna(refs) or not str(refs).strip():
            continue

        reference_ids = {
            ref.strip()
            for ref in str(refs).split(";")
            if ref.strip()
        }

        matches = results[
            (results["query_id"] == query["query_id"])
            & (results["project_id"].isin(reference_ids))
        ]

        best_rank = (
            int(matches["rank"].min())
            if not matches.empty else None
        )

        records.append({
            "query_id": query["query_id"],
            "language": query["language"],
            "best_reference_rank": best_rank,
            "reciprocal_rank": (
                1 / best_rank if best_rank is not None else 0.0
            )
        })

    return pd.DataFrame(records)


original_ranks = reference_rank_analysis(
    original_batch_results, dev_queries
)

chunked_ranks = reference_rank_analysis(
    chunked_batch_results, dev_queries
)

rank_comparison = pd.concat([
    original_ranks.assign(method="Original"),
    chunked_ranks.assign(method="Chunked")
])

rank_summary = (
    rank_comparison
    .groupby(["language", "method"])
    .agg(
        queries=("query_id", "count"),
        mrr_at_5=("reciprocal_rank", "mean")
    )
    .reset_index()
)

display(rank_summary.round(4))

,language,method,queries,mrr_at_5
0,ar,Chunked,13,0.7295
1,ar,Original,13,0.6756
2,en,Chunked,13,0.7115
3,en,Original,13,0.7179


## 5.10. Query-Level Ranking Analysis

Identify development queries whose reference-project rankings improved, declined, or remained unchanged after chunking.

In [27]:
# Compare reciprocal ranks per query
rank_changes = original_ranks.merge(
    chunked_ranks,
    on=["query_id", "language"],
    suffixes=("_original", "_chunked")
)

rank_changes["mrr_change"] = (
    rank_changes["reciprocal_rank_chunked"]
    - rank_changes["reciprocal_rank_original"]
)

rank_changes["status"] = np.select(
    [
        rank_changes["mrr_change"] > 1e-9,
        rank_changes["mrr_change"] < -1e-9
    ],
    ["Improved", "Declined"],
    default="Unchanged"
)

print("Ranking changes by language:")
display(
    pd.crosstab(
        rank_changes["language"],
        rank_changes["status"]
    )
)

print("\nQueries with ranking changes:")
display(
    rank_changes.loc[
        rank_changes["status"] != "Unchanged",
        [
            "query_id",
            "language",
            "best_reference_rank_original",
            "best_reference_rank_chunked",
            "mrr_change",
            "status"
        ]
    ].sort_values("mrr_change", ascending=False).round(4)
)

Ranking changes by language:


status,Declined,Improved,Unchanged
language,,,
ar,4,3,6
en,1,1,11



Queries with ranking changes:


,query_id,language,best_reference_rank_original,best_reference_rank_chunked,mrr_change,status
25,Q049,ar,5.0,1.0,0.8000,Improved
8,Q018,ar,4.0,1.0,0.7500,Improved
9,Q019,ar,4.0,2.0,0.2500,Improved
6,Q014,en,3.0,2.0,0.1667,Improved
23,Q046,ar,4.0,5.0,-0.0500,Declined
10,Q020,ar,3.0,4.0,-0.0833,Declined
5,Q013,en,2.0,4.0,-0.2500,Declined
2,Q006,ar,2.0,5.0,-0.3000,Declined
19,Q040,ar,1.0,3.0,-0.6667,Declined


## 5.11. Query Token Length Diagnostic

Examine development query lengths to identify potential truncation during semantic query encoding.

In [28]:
# Count query tokens without truncation
query_token_counts = [
    len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )
    for text in dev_queries["query_text"]
]

query_diagnostics = dev_queries[
    ["query_id", "language"]
].copy()

query_diagnostics["token_count"] = query_token_counts

query_diagnostics["is_truncated"] = (
    query_diagnostics["token_count"] > model.max_seq_length
)

print("QUERY TOKEN LENGTH DIAGNOSTIC")
print("=" * 50)
print("Model maximum tokens:", model.max_seq_length)
print("Total queries:", len(query_diagnostics))
print("Queries exceeding limit:", query_diagnostics["is_truncated"].sum())

print("\nTruncation by language:")
display(
    query_diagnostics.groupby("language")["is_truncated"]
    .agg(["count", "sum", "mean"])
    .round(4)
)

print("\nTop 10 longest queries:")
display(
    query_diagnostics.sort_values(
        "token_count",
        ascending=False
    ).head(10)
)

QUERY TOKEN LENGTH DIAGNOSTIC
Model maximum tokens: 128
Total queries: 40
Queries exceeding limit: 17

Truncation by language:


,count,sum,mean
language,,,
ar,20,12,0.60
en,20,5,0.25



Top 10 longest queries:


,query_id,language,token_count,is_truncated
25,Q039,ar,187,True
15,Q027,ar,185,True
39,Q060,ar,181,True
17,Q030,ar,179,True
8,Q018,ar,178,True
14,Q026,ar,165,True
22,Q036,ar,165,True
2,Q006,ar,163,True
33,Q051,en,149,True
37,Q056,ar,148,True


## 5.12. Query Chunking and Embedding Generation

Split development queries into token-safe chunks and generate normalized query embeddings using mean pooling.

In [29]:
# Chunk development queries
query_chunk_texts = [
    safe_chunk_text(text)
    for text in dev_queries["query_text"]
]

all_query_chunks = [
    chunk
    for chunks in query_chunk_texts
    for chunk in chunks
]

# Validate chunk lengths
assert all(
    len(model.tokenizer.encode(
        chunk,
        add_special_tokens=True,
        truncation=False
    )) <= model.max_seq_length
    for chunk in all_query_chunks
)

# Generate query chunk embeddings
query_chunk_embeddings = model.encode(
    all_query_chunks,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Aggregate embeddings per query
chunked_query_embeddings = []
start = 0

for chunks in query_chunk_texts:
    count = len(chunks)
    vectors = query_chunk_embeddings[start:start + count]

    mean_vector = vectors.mean(axis=0)
    norm = np.linalg.norm(mean_vector)

    assert norm > 0
    chunked_query_embeddings.append(mean_vector / norm)

    start += count

chunked_query_embeddings = np.vstack(chunked_query_embeddings)

# Validate results
assert start == len(query_chunk_embeddings)
assert chunked_query_embeddings.shape == (40, 768)
assert np.isfinite(chunked_query_embeddings).all()
assert np.allclose(
    np.linalg.norm(chunked_query_embeddings, axis=1),
    1.0,
    atol=1e-5
)

print("Query chunking completed successfully!")
print("Total queries:", len(query_chunk_texts))
print("Total query chunks:", len(all_query_chunks))
print("Queries requiring multiple chunks:",
      sum(len(chunks) > 1 for chunks in query_chunk_texts))
print("Embeddings shape:", chunked_query_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Query chunking completed successfully!
Total queries: 40
Total query chunks: 57
Queries requiring multiple chunks: 17
Embeddings shape: (40, 768)


## 5.13. Retrieval Comparison with Chunked Queries

Re-evaluate original and chunked document embeddings using the same chunked query representations and Cosine Similarity.

In [30]:
# Calculate cosine similarity using chunked queries
original_similarities_full = cosine_similarity(
    chunked_query_embeddings,
    project_embeddings
)

chunked_similarities_full = cosine_similarity(
    chunked_query_embeddings,
    chunked_project_embeddings
)

# Retrieve Top-5 for both approaches
original_batch_results_full = batch_retrieve(
    original_similarities_full,
    "original_full_queries"
)

chunked_batch_results_full = batch_retrieve(
    chunked_similarities_full,
    "chunked_full_queries"
)

# Validate retrieval results
assert original_similarities_full.shape == (40, 103)
assert chunked_similarities_full.shape == (40, 103)

assert len(original_batch_results_full) == 200
assert len(chunked_batch_results_full) == 200

assert original_batch_results_full.groupby("query_id").size().eq(5).all()
assert chunked_batch_results_full.groupby("query_id").size().eq(5).all()

print("Retrieval comparison completed successfully!")
print("Original results:", len(original_batch_results_full))
print("Chunked results:", len(chunked_batch_results_full))

display(chunked_batch_results_full.head(10).round(4))

Retrieval comparison completed successfully!
Original results: 200
Chunked results: 200


,query_id,language,method,rank,project_id,similarity_score
0,Q003,en,chunked_full_queries,1,P059,0.7494
1,Q003,en,chunked_full_queries,2,P083,0.5754
2,Q003,en,chunked_full_queries,3,P017,0.4933
3,Q003,en,chunked_full_queries,4,P070,0.4922
4,Q003,en,chunked_full_queries,5,P082,0.4920
5,Q004,en,chunked_full_queries,1,P074,0.6598
6,Q004,en,chunked_full_queries,2,P003,0.5552
7,Q004,en,chunked_full_queries,3,P036,0.5506
8,Q004,en,chunked_full_queries,4,P069,0.4527
9,Q004,en,chunked_full_queries,5,P085,0.3705


## 5.14. Full-Query Retrieval Evaluation

Compare original and chunked document embeddings using fully processed development queries and reference-based Hit@5 and MRR@5.

In [31]:
# Evaluate reference retrieval
original_hits_full = reference_hit_analysis(
    original_batch_results_full, dev_queries
)

chunked_hits_full = reference_hit_analysis(
    chunked_batch_results_full, dev_queries
)

# Evaluate reference rankings
original_ranks_full = reference_rank_analysis(
    original_batch_results_full, dev_queries
)

chunked_ranks_full = reference_rank_analysis(
    chunked_batch_results_full, dev_queries
)

# Combine evaluation results
evaluation = pd.concat([
    original_hits_full.merge(
        original_ranks_full,
        on=["query_id", "language"]
    ).assign(method="Original"),

    chunked_hits_full.merge(
        chunked_ranks_full,
        on=["query_id", "language"]
    ).assign(method="Chunked")
])

# Summarize results by language
summary_full = (
    evaluation
    .groupby(["language", "method"])
    .agg(
        queries=("query_id", "count"),
        hits=("reference_hit_at_5", "sum"),
        hit_at_5=("reference_hit_at_5", "mean"),
        mrr_at_5=("reciprocal_rank", "mean")
    )
    .reset_index()
)

display(summary_full.round(4))

print("Full-query evaluation completed successfully!")

,language,method,queries,hits,hit_at_5,mrr_at_5
0,ar,Chunked,13,12,0.9231,0.6141
1,ar,Original,13,12,0.9231,0.5564
2,en,Chunked,13,11,0.8462,0.7500
3,en,Original,13,11,0.8462,0.7179


Full-query evaluation completed successfully!


## 5.15. Save Chunking Experiment Results

Save original and chunked semantic retrieval results, evaluation summaries, and chunked project embeddings for reproducibility.

---



In [32]:
from pathlib import Path

# Create experiment output directory
chunking_output_dir = Path("/content/origin_chunking_outputs")
chunking_output_dir.mkdir(parents=True, exist_ok=True)

# Save retrieval results
original_batch_results_full.to_csv(
    chunking_output_dir / "original_dev_top5.csv",
    index=False
)

chunked_batch_results_full.to_csv(
    chunking_output_dir / "chunked_dev_top5.csv",
    index=False
)

# Save evaluation summary
summary_full.to_csv(
    chunking_output_dir / "chunking_evaluation_summary.csv",
    index=False
)

# Save chunked project embeddings
np.save(
    chunking_output_dir / "chunked_project_embeddings.npy",
    chunked_project_embeddings
)

# Preserve embedding-to-project mapping
semantic_df[["project_id"]].to_csv(
    chunking_output_dir / "embedding_project_ids.csv",
    index=False
)

print("Chunking experiment results saved successfully!")

for file in sorted(chunking_output_dir.iterdir()):
    print(file.name)

Chunking experiment results saved successfully!
chunked_dev_top5.csv
chunked_project_embeddings.npy
chunking_evaluation_summary.csv
embedding_project_ids.csv
original_dev_top5.csv


In [33]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_chunking_outputs",
    "zip",
    chunking_output_dir
)

files.download(archive_path)

print("Chunking experiment outputs downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Chunking experiment outputs downloaded successfully!


## 6. Semantic Similarity and Top-K Retrieval

Retrieve the Top-K most semantically related graduation projects using Cosine Similarity.

In [34]:
def semantic_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    if not 1 <= top_k <= len(semantic_df):
        raise ValueError("Invalid top_k value.")

    # Prepare and chunk query
    query_text = f"{title.strip()}. {description.strip()}"
    query_chunks = safe_chunk_text(query_text)

    # Generate query chunk embeddings
    query_vectors = model.encode(
        query_chunks,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Aggregate query embeddings
    query_embedding = query_vectors.mean(axis=0, keepdims=True)
    query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True)

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_embedding,
        chunked_project_embeddings
    )[0]

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = semantic_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("Chunked semantic search function is ready!")

Chunked semantic search function is ready!


## 7. Semantic Search Testing

Test semantic retrieval using an English project idea and inspect the Top-5 results.

In [35]:
# Test English query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

english_results = semantic_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    english_results[
        ["rank", "project_id", "title", "similarity_score"]
    ].round(4)
)

,rank,project_id,title,similarity_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,0.8100
1,2,P080,A Smart Mobile Application For Skin Cancer Det...,0.5332
2,3,P005,Thabat – ثبات,0.5147
3,4,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,0.5130
4,5,P082,Predicting Diabetes Patients Based On Machine ...,0.4944


In [36]:
# Test Arabic query
arabic_title = "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"

arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

arabic_results = semantic_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_results[
        ["rank", "project_id", "title", "similarity_score"]
    ].round(4)
)

,rank,project_id,title,similarity_score
0,1,P082,Predicting Diabetes Patients Based On Machine ...,0.6584
1,2,P025,Early Detection and Monitoring of Parkinson’s ...,0.6342
2,3,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,0.5788
3,4,P080,A Smart Mobile Application For Skin Cancer Det...,0.5604
4,5,P093,Deep Learning Based Wound Classification And T...,0.5512


## 8. Cross-Lingual Retrieval Analysis

Compare the Top-5 retrieved projects for equivalent English and Arabic queries.

In [38]:
comparison = pd.merge(
    english_results[["project_id", "rank", "similarity_score"]],
    arabic_results[["project_id", "rank", "similarity_score"]],
    on="project_id",
    how="outer",
    suffixes=("_en", "_ar")
)

comparison = comparison.sort_values(
    ["rank_en", "rank_ar"],
    na_position="last"
)

display(comparison.round(4))

overlap = len(
    set(english_results["project_id"])
    & set(arabic_results["project_id"])
)

print(f"Top-5 overlap: {overlap}/5")

,project_id,rank_en,similarity_score_en,rank_ar,similarity_score_ar
1,P025,1.0,0.8100,2.0,0.6342
3,P080,2.0,0.5332,4.0,0.5604
0,P005,3.0,0.5147,NaN,NaN
2,P032,4.0,0.5130,3.0,0.5788
4,P082,5.0,0.4944,1.0,0.6584
5,P093,NaN,NaN,5.0,0.5512


Top-5 overlap: 4/5


## 9. Retrieval Validation

Verify embedding normalization, similarity scores, and retrieval ranking.

In [39]:
# Validate chunked project embeddings
assert chunked_project_embeddings.shape == (
    len(semantic_df),
    model.get_sentence_embedding_dimension()
)

assert np.isfinite(chunked_project_embeddings).all()

assert np.allclose(
    np.linalg.norm(chunked_project_embeddings, axis=1),
    1.0,
    atol=1e-5
)

# Validate retrieval results
for results in [english_results, arabic_results]:
    assert len(results) == 5
    assert results["project_id"].is_unique
    assert results["similarity_score"].between(-1, 1).all()
    assert results["similarity_score"].is_monotonic_decreasing
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]

# Validate input handling
for invalid_title, invalid_description in [
    ("", "Valid description"),
    ("Valid title", ""),
]:
    try:
        semantic_search(invalid_title, invalid_description)
        raise AssertionError("Empty input was accepted.")
    except ValueError:
        pass

print("All chunked semantic retrieval validation checks passed!")

All chunked semantic retrieval validation checks passed!


/tmp/ipykernel_424/3585711339.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 10. Save Results and Embeddings

Save the English and Arabic retrieval results and the generated project embeddings for future evaluation and reuse.

In [42]:
from pathlib import Path
import numpy as np

# Create output directory
output_dir = Path("/content/origin_semantic_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Remove old embeddings file from output folder
(output_dir / "project_embeddings.npy").unlink(missing_ok=True)

# Save updated retrieval results
english_results.to_csv(
    output_dir / "semantic_english_results.csv",
    index=False,
    encoding="utf-8-sig"
)

arabic_results.to_csv(
    output_dir / "semantic_arabic_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save chunked project embeddings
np.save(
    output_dir / "chunked_project_embeddings.npy",
    chunked_project_embeddings
)

# Save project ID mapping
semantic_df[["project_id"]].to_csv(
    output_dir / "embedding_project_ids.csv",
    index=False
)

print("Chunked semantic search outputs saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)

Chunked semantic search outputs saved successfully!
chunked_project_embeddings.npy
embedding_project_ids.csv
semantic_arabic_results.csv
semantic_english_results.csv


In [43]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_semantic_outputs",
    "zip",
    output_dir
)

files.download(archive_path)

print("Semantic search outputs downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Semantic search outputs downloaded successfully!



## 11. Conclusions

This notebook implemented multilingual semantic retrieval for graduation projects using a pretrained Sentence Transformer and Cosine Similarity.

### Key Findings

- The initial model successfully retrieved relevant historical projects using Arabic and English queries.
- Token length diagnostics revealed that 98 out of 103 project descriptions (95.15%) exceeded the model's 128-token limit.
- Token-based chunking with 20-token overlap was implemented to preserve information from longer project descriptions.
- Chunk embeddings were aggregated using mean pooling and L2 normalization to produce one embedding per project.
- Query chunking was also implemented after identifying 17 out of 40 development queries exceeding the model's token limit.
- The updated `semantic_search()` function uses chunked representations for both project descriptions and incoming queries.

### Development Evaluation Results

The original and chunked document representations were compared using the same chunked query embeddings on 40 development queries.

| Metric | Original Documents | Chunked Documents |
|---|---:|---:|
| Arabic Reference Hit@5 | 0.9231 | 0.9231 |
| English Reference Hit@5 | 0.8462 | 0.8462 |
| Arabic Reference MRR@5 | 0.5564 | 0.6141 |
| English Reference MRR@5 | 0.7179 | 0.7500 |

The chunked approach improved reference-based ranking performance in both languages while maintaining the same Reference Hit@5.

These metrics were calculated using 26 development queries (13 per language) with human-selected reference project IDs. The reference projects were identified through team review. However, the labels may not cover every relevant project in the dataset.

### Limitations and Future Work

- Mean pooling may not be the optimal strategy for aggregating chunk embeddings.
- Chunking improved average reference-based ranking metrics but did not improve every individual query.
- Further evaluation is needed to assess retrieval performance beyond the selected reference projects.
- The selected semantic retrieval approach will be integrated into Hybrid Search and compared against TF-IDF and BM25-based retrieval.
- Final model selection should be based on evaluation results from the reserved test set.

The chunked semantic approach is the current development candidate for Origin, pending further evaluation and integration.
